In [1]:
import os
import json
import random
import shutil

seed = 42
base_image_dir = "/data_center/data2/dataset/chenwy/21164-data/coco_2014/val2014"
target_image_dir = "/data_center/data2/dataset/chenwy/21164-data/coco_2014/val2014-10k"
base_json_path = "/data_center/data2/dataset/chenwy/21164-data/coco_2014/mscoco_val_2014.json"
target_json_path = base_json_path.replace('.json', '_10k.json')
os.makedirs(target_image_dir, exist_ok=True)

random.seed(seed)

# 读取 JSON 文件
with open(base_json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# 随机选择 10k 条数据
# 生成一个长度为 len(data) 的随机索引序列
indices = list(range(len(data)))
random.shuffle(indices)

# 选择前 10k 个索引
sample_size = min(10000, len(data))
selected_indices = indices[:sample_size]

# 根据索引选择对应的数据
sampled_data = [data[i] for i in selected_indices]
print(f"总共有 {len(data)} 条数据")
print(f"随机选择了 {sample_size} 条数据")

# 保存采样后的 JSON 数据（可选）
with open(target_json_path, 'w', encoding='utf-8') as f:
    json.dump(sampled_data, f, ensure_ascii=False, indent=2)
print(f"已保存采样后的 JSON 到: {target_json_path}")

# 创建符号链接
linked_count = 0
missing_count = 0

for item in sampled_data:
    # 获取图片文件名（image 字段可能包含完整路径、文件名或只是 ID）
    image_field = item.get('image', '')
    
    # 如果 image 字段是完整路径，提取文件名
    if '/' in image_field:
        image_filename = os.path.basename(image_field)
    elif image_field.isdigit() or (image_field.lstrip('0').isdigit() and image_field.startswith('0')):
        # 如果 image 字段是纯数字 ID，构建 COCO 格式的文件名
        # 例如: 581929 -> COCO_val2014_000000581929.jpg
        image_id = int(image_field.lstrip('0') or '0')  # 处理全0的情况
        image_filename = f"COCO_val2014_{image_id:012d}.jpg"
    else:
        # 如果已经是文件名，直接使用
        image_filename = image_field
    
    # 构建源图片路径
    source_image_path = os.path.join(base_image_dir, image_filename)
    target_image_path = os.path.join(target_image_dir, image_filename)
    
    # 检查源文件是否存在
    if os.path.exists(source_image_path):
        # 如果目标链接已存在，先删除
        if os.path.exists(target_image_path) or os.path.islink(target_image_path):
            os.remove(target_image_path)
        
        # 创建符号链接
        os.symlink(source_image_path, target_image_path)
        linked_count += 1
    else:
        print(f"警告: 源图片不存在: {source_image_path}")
        missing_count += 1

print(f"\n完成！")
print(f"成功创建符号链接: {linked_count} 个")
print(f"缺失的图片: {missing_count} 个")
print(f"目标目录: {target_image_dir}")

总共有 40371 条数据
随机选择了 10000 条数据
已保存采样后的 JSON 到: /data_center/data2/dataset/chenwy/21164-data/coco_2014/mscoco_val_2014_10k.json

完成！
成功创建符号链接: 10000 个
缺失的图片: 0 个
目标目录: /data_center/data2/dataset/chenwy/21164-data/coco_2014/val2014-10k
